# OPTN Kidney Waitlist Data Processing

This notebook creates a cleaned and model-ready feature matrix from the OPTN kidney waitlist analytic extract.

In [1]:
# Import the libraries used throughout the processing pipeline

from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler


In [2]:
# Load the source extract with identifier-like and mixed-type fields preserved safely

raw_df = pd.read_csv(
    "../../data/restricted/kidney_waitlist_analytic.csv.gz",
    low_memory=False,
)

df = raw_df.copy()


In [3]:
# Define the listing-time feature set and the modeling target from the data dictionary

target_col = "event_adverse"

feature_cols = [
    "ON_DIALYSIS",
    "A2A2B_ELIGIBILITY",
    "GENDER",
    "ABO",
    "BMI_TCR",
    "FUNC_STAT_TCR",
    "INIT_STAT",
    "INIT_CPRA",
    "INIT_AGE",
    "DIALYSIS_DATE",
    "INIT_DATE",
    "ETHCAT",
    "REGION",
]

required_cols = feature_cols + [target_col]
missing_required = sorted(set(required_cols) - set(df.columns))
if missing_required:
    raise KeyError(f"Required columns are missing from the source extract: {missing_required}")

df_model = df[required_cols].copy()


In [4]:
# Parse listing and dialysis dates for time-based feature derivation

df_model["INIT_DATE"] = pd.to_datetime(df_model["INIT_DATE"], errors="coerce")
df_model["DIALYSIS_DATE"] = pd.to_datetime(df_model["DIALYSIS_DATE"], errors="coerce")


In [5]:
# Create missingness indicators before imputing BMI and cPRA

df_model["bmi_missing"] = df_model["BMI_TCR"].isna().astype("int8")
df_model["cpra_missing"] = df_model["INIT_CPRA"].isna().astype("int8")


In [6]:
# Replace implausible BMI values outside the documented 10–80 kg/m² range with the valid-value median

bmi_min, bmi_max = 10, 80
bmi_valid = df_model["BMI_TCR"].between(bmi_min, bmi_max)
bmi_median = df_model.loc[bmi_valid, "BMI_TCR"].median()
df_model.loc[~bmi_valid, "BMI_TCR"] = bmi_median


In [7]:
# Impute missing cPRA and preserve its missingness indicator

df_model["INIT_CPRA"] = df_model["INIT_CPRA"].fillna(df_model["INIT_CPRA"].median())


In [8]:
# Decode functional-status codes into scale type and percentage while retaining missing or unresolved cases

functional_scale_map = {-1: "missing", 996: "not_applicable", 998: "unknown", 1: "adl_independent"}
functional_percent_map = {-1: np.nan, 996: np.nan, 998: np.nan, 1: np.nan}

for code in range(2010, 2101, 10):
    functional_scale_map[code] = "adult_karnofsky"
    functional_percent_map[code] = code - 2000

for code in range(4010, 4101, 10):
    functional_scale_map[code] = "pediatric_lansky"
    functional_percent_map[code] = code - 4000

functional_codes = pd.to_numeric(df_model["FUNC_STAT_TCR"], errors="coerce")
df_model["functional_scale"] = functional_codes.map(functional_scale_map).fillna("other_unresolved")
df_model["functional_percent"] = functional_codes.map(functional_percent_map)
df_model["functional_percent"] = df_model["functional_percent"].fillna(
    df_model["functional_percent"].median()
)
df_model = df_model.drop(columns="FUNC_STAT_TCR")


In [9]:
# Calculate the number of dialysis days completed before initial listing

df_model["dialysis_duration_days"] = (
    df_model["INIT_DATE"] - df_model["DIALYSIS_DATE"]
).dt.days
df_model["dialysis_duration_days"] = df_model["dialysis_duration_days"].fillna(0).clip(lower=0)


In [10]:
# Create policy-era and BMI-category features for temporal and nonlinear effects

def policy_era(listing_date):
    if pd.isna(listing_date):
        return "unknown"
    if listing_date < pd.Timestamp("2015-01-01"):
        return "pre_2015"
    if listing_date < pd.Timestamp("2021-01-01"):
        return "2015_2020"
    return "post_2021"


df_model["policy_era"] = df_model["INIT_DATE"].map(policy_era)
df_model["bmi_category"] = pd.cut(
    df_model["BMI_TCR"],
    bins=[0, 18.5, 25, 30, np.inf],
    labels=["underweight", "normal", "overweight", "obese"],
)
df_model["age_x_functional"] = df_model["INIT_AGE"] * df_model["functional_percent"]


In [11]:
# Prepare categorical values, treating missing categorical data as an explicit unknown category

categorical_cols = [
    "ON_DIALYSIS",
    "GENDER",
    "ABO",
    "A2A2B_ELIGIBILITY",
    "INIT_STAT",
    "ETHCAT",
    "REGION",
    "functional_scale",
    "policy_era",
    "bmi_category",
]

for col in categorical_cols:
    df_model[col] = df_model[col].astype("string").fillna("unknown")


In [12]:
# Impute any remaining numeric feature gaps with training-data medians

numerical_cols = [
    "BMI_TCR",
    "INIT_CPRA",
    "INIT_AGE",
    "functional_percent",
    "dialysis_duration_days",
    "age_x_functional",
]

for col in numerical_cols:
    df_model[col] = df_model[col].fillna(df_model[col].median())


In [13]:
# Keep the cleaned, human-readable dataset in memory before encoding

cleaned_df = df_model.drop(columns=["DIALYSIS_DATE", "INIT_DATE"])


In [14]:
# Separate the target from the listing-time predictors to prevent outcome leakage

y = df_model[target_col].astype("int8")
X = df_model.drop(columns=[target_col, "DIALYSIS_DATE", "INIT_DATE"])


In [15]:
# One-hot encode categorical predictors while retaining a reference level for each category

X_encoded = pd.get_dummies(
    X,
    columns=categorical_cols,
    drop_first=True,
    dtype="int8",
)


In [16]:
# Standardize continuous predictors for models that are sensitive to feature scale

scaler = StandardScaler()
X_encoded[numerical_cols] = scaler.fit_transform(X_encoded[numerical_cols])


In [17]:
# Combine the encoded predictors with the target for downstream modeling

engineered_df = X_encoded.copy()
engineered_df[target_col] = y.to_numpy()


In [18]:
# Store compact validation results for row-count, missing-value, and target checks

validation = {
    "source_rows": len(raw_df),
    "cleaned_rows": len(cleaned_df),
    "engineered_rows": len(engineered_df),
    "engineered_columns": engineered_df.shape[1],
    "missing_engineered_values": int(engineered_df.isna().sum().sum()),
    "target_values": sorted(engineered_df[target_col].unique().tolist()),
}

validation


{'source_rows': 494862,
 'cleaned_rows': 494862,
 'engineered_rows': 494862,
 'engineered_columns': 50,
 'missing_engineered_values': 0,
 'target_values': [0, 1]}

In [19]:
# Export the final in-memory processed dataset as the only file created by the notebook

OUTPUT_PATH = Path("cleaned_data_final.csv")
engineered_df.to_csv(OUTPUT_PATH, index=False)
